# Tratamento dos Dados — Mamografias


---
## 1. Imports

In [1]:
import re
import unicodedata
from pathlib import Path

import pandas as pd
from IPython.display import display, Markdown

---
## 2. Configurações Gerais


In [2]:
# ── Caminhos ─────────────────────────────────────────────────────────────────
ENTRADA   = Path("/home/anna/tcc/Limpeza e Integração/Mamografia/Dados")
SAIDA     = Path("/home/anna/tcc/Seleção e Transformação/Mamografias/tabela_mamografia.csv")
MUNICIPIOS = Path("/home/anna/tcc/Seleção e Transformação/Municípios/tabela_municipios_regionais.csv")

# ── Faixas etárias padronizadas e seus padrões de origem ─────────────────────
# Cada tupla: (nome_da_faixa_destino, [padrões_de_texto_que_identificam_a_faixa_origem])
# Os padrões são buscados no nome normalizado das colunas do CSV bruto.
FAIXAS: list[tuple[str, list[str]]] = [
    ("0-9",       ["ate 9"]),
    ("10-19",     ["entre 10 a 14", "entre 15 a 19"]),
    ("20-29",     ["entre 20 a 24", "entre 25 a 29"]),
    ("30-39",     ["entre 30 a 34", "entre 35 a 39"]),
    ("40-49",     ["entre 40 a 44", "entre 45 a 49"]),
    ("50-59",     ["entre 50 a 54", "entre 55 a 59"]),
    ("60-69",     ["entre 60 a 64", "entre 65 a 69"]),
    ("70 OU MAIS", ["entre 70 a 74", "entre 75 a 79", "acima de 79"]),
]

SAIDA.parent.mkdir(parents=True, exist_ok=True)
print(f"✅ Pasta de saída: {SAIDA.parent}")
print(f"   Faixas configuradas: {[f for f, _ in FAIXAS]}")

✅ Pasta de saída: /home/anna/tcc/Seleção e Transformação/Mamografias
   Faixas configuradas: ['0-9', '10-19', '20-29', '30-39', '40-49', '50-59', '60-69', '70 OU MAIS']


---
## 3. Funções de Normalização de Texto

Funções para padronizar nomes de municípios e colunas dos CSVs brutos.

| Função | Responsabilidade |
|---|---|
| `remover_acentos` | Remove diacríticos via NFD |
| `normalizar_municipio` | Strip + lowercase + remove caracteres especiais |
| `normalizar_coluna` | Igual a `normalizar_municipio`, mas garante retorno string (para nomes de colunas) |
| `extrair_ano_do_nome_arquivo` | Extrai o ano (19xx/20xx) do nome do arquivo |
| `extrair_codigo_municipio` | Extrai código IBGE e nome do município da primeira coluna do CSV bruto |

In [3]:
def remover_acentos(texto: str) -> str:
    """Remove todos os diacríticos (acentos, cedilha etc.) de uma string."""
    normalizado = unicodedata.normalize("NFD", texto)
    return "".join(c for c in normalizado if unicodedata.category(c) != "Mn")


def normalizar_municipio(valor: object) -> object:
    """Strip + lowercase + sem acentos + remove caracteres não alfanuméricos.
    Retorna o valor original se não for string."""
    if not isinstance(valor, str):
        return valor
    texto = remover_acentos(valor).lower().strip()
    texto = re.sub(r"[^0-9a-z]+", " ", texto)
    return " ".join(texto.split())


def normalizar_coluna(texto: str) -> str:
    """Normaliza nome de coluna. Retorna string vazia se o resultado não for string."""
    resultado = normalizar_municipio(texto)
    return "" if not isinstance(resultado, str) else resultado


def extrair_ano_do_nome_arquivo(path: Path) -> int:
    """Extrai o ano (19xx ou 20xx) do nome do arquivo. Lança ValueError se não encontrar."""
    m = re.search(r"(19|20)\d{2}", path.stem)
    if not m:
        raise ValueError(f"Ano não encontrado no nome do arquivo: {path.name}")
    return int(m.group(0))


def extrair_codigo_municipio(valor: object) -> tuple:
    """
    Extrai código IBGE e nome do município da primeira coluna do CSV bruto.
    Formato esperado: '1234567 Nome do Município'
    Retorna (codigo, municipio) ou (None, None) se não casar.
    """
    if not isinstance(valor, str):
        return None, None
    texto = valor.strip().strip("\ufeff")
    m = re.match(r"\s*(\d{1,7})\s+(.*)$", texto)
    if not m:
        return None, None
    return m.group(1), m.group(2)


print("✅ Funções de normalização definidas.")

✅ Funções de normalização definidas.


---
## 4. Função de Carregamento por Ano

`carregar_por_ano` lê um único arquivo CSV anual e:
1. Extrai código IBGE e nome do município da primeira coluna
2. Agrupa as colunas brutas nas 8 faixas etárias padronizadas
3. Agrega por município (soma) e renomeia as colunas para `YYYY_faixa`

In [4]:
def carregar_por_ano(arquivo: Path) -> pd.DataFrame:
    """
    Lê um CSV anual de mamografias (sep=';', latin1) e retorna um DataFrame
    com colunas: codigo, municipios, YYYY_0-9, YYYY_10-19, …, YYYY_70 OU MAIS
    """
    ano    = extrair_ano_do_nome_arquivo(arquivo)
    tabela = pd.read_csv(arquivo, sep=";", encoding="latin1", dtype=str)
    tabela.columns = tabela.columns.str.strip()

    # Extrai código e nome do município da primeira coluna
    col_municipio = tabela.columns[0]
    extraido = tabela[col_municipio].map(extrair_codigo_municipio)
    tabela[["codigo", "municipio_raw"]] = pd.DataFrame(extraido.tolist(), index=tabela.index)
    tabela = tabela.dropna(subset=["codigo", "municipio_raw"]).copy()

    tabela["codigo"]     = tabela["codigo"].astype(str).str.strip().str.zfill(7)
    tabela["municipios"] = tabela["municipio_raw"].map(normalizar_municipio)
    tabela = tabela.drop(columns=["municipio_raw"])

    # Mapa de nome de coluna normalizado → nome original
    mapa_colunas = {normalizar_coluna(c): c for c in tabela.columns}

    saida = tabela[["codigo", "municipios"]].copy()

    # Agrupa colunas brutas nas faixas padronizadas
    for faixa, padroes in FAIXAS:
        colunas_origem = sorted({
            col_original
            for p in padroes
            for chave_norm, col_original in mapa_colunas.items()
            if p in chave_norm
        })

        if not colunas_origem:
            saida[faixa] = 0
            continue

        saida[faixa] = (
            tabela[colunas_origem]
            .apply(lambda s: pd.to_numeric(s, errors="coerce"), axis=0)
            .fillna(0)
            .sum(axis=1)
            .astype(int)
        )

    # Agrega por município (pode haver duplicatas por subdivisão)
    colunas_faixas = [f for f, _ in FAIXAS]
    saida = (
        saida.groupby(["codigo", "municipios"], as_index=False)[colunas_faixas]
        .sum()
        .reset_index(drop=True)
    )

    # Renomeia para YYYY_faixa
    return saida.rename(columns={c: f"{ano}_{c}" for c in colunas_faixas})


print("✅ carregar_por_ano definida.")

✅ carregar_por_ano definida.


---
## 5. Pipeline de Tratamento

Executa o pipeline completo em 4 etapas:
1. Localiza todos os arquivos `mamografia_YYYY.csv`
2. Carrega e une ano a ano via `outer merge` pelo código do município
3. Calcula as colunas de total por ano (`YYYY_TOTAL`), agrupadas ao final da tabela
4. Enriquece com código IBGE completo (7 dígitos) e `REGIONAL DE SAÚDE` via tabela de municípios

In [5]:
# ── Etapa 1: Localizar arquivos ───────────────────────────────────────────────
print("🔍 Localizando arquivos brutos...")
arquivos = sorted(ENTRADA.glob("mamografia_*.csv"))
if not arquivos:
    raise FileNotFoundError(f"Nenhum arquivo encontrado em: {ENTRADA}")

anos_encontrados = [extrair_ano_do_nome_arquivo(a) for a in arquivos]
print(f"   {len(arquivos)} arquivo(s): {anos_encontrados}")

🔍 Localizando arquivos brutos...
   13 arquivo(s): [2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]


In [6]:
# ── Etapa 2: Carregar e unir ano a ano ───────────────────────────────────────
print("\n⚙️  Carregando e unindo anos...")
tabela: pd.DataFrame | None = None

for arquivo in arquivos:
    df_ano = carregar_por_ano(arquivo)
    ano    = extrair_ano_do_nome_arquivo(arquivo)

    if tabela is None:
        tabela = df_ano
        print(f"   {ano}: {len(df_ano)} municípios (base)")
        continue

    tabela = tabela.merge(df_ano, on=["codigo"], how="outer", suffixes=("", "_dup"))

    # Consolida coluna de município (outer join pode gerar duplicata)
    if "municipios_dup" in tabela.columns:
        tabela["municipios"] = tabela["municipios"].fillna(tabela["municipios_dup"])
        tabela = tabela.drop(columns=["municipios_dup"])

    print(f"   {ano}: {len(df_ano)} municípios → tabela acumulada: {len(tabela)}")

assert tabela is not None, "Nenhum arquivo foi processado."

# Ordena colunas: municipios, codigo, depois YYYY_faixa por ano e faixa
ordem_faixas = [f for f, _ in FAIXAS]
anos = sorted({
    int(c.split("_")[0])
    for c in tabela.columns
    if re.match(r"^(19|20)\d{2}_", str(c))
})
colunas_ano = [
    f"{ano}_{faixa}"
    for ano in anos
    for faixa in ordem_faixas
    if f"{ano}_{faixa}" in tabela.columns
]
tabela = tabela[["municipios", "codigo"] + colunas_ano]

# Garante tipo int em todas as colunas de faixa
tabela[colunas_ano] = (
    tabela[colunas_ano]
    .apply(lambda s: pd.to_numeric(s, errors="coerce"), axis=0)
    .fillna(0)
    .astype(int)
)

print(f"\n✅ Tabela base: {tabela.shape}")


⚙️  Carregando e unindo anos...
   2013: 4 municípios (base)
   2014: 521 municípios → tabela acumulada: 521
   2015: 409 municípios → tabela acumulada: 583
   2016: 778 municípios → tabela acumulada: 813
   2017: 812 municípios → tabela acumulada: 840
   2018: 814 municípios → tabela acumulada: 846
   2019: 802 municípios → tabela acumulada: 849


   2020: 775 municípios → tabela acumulada: 849


   2021: 792 municípios → tabela acumulada: 850
   2022: 784 municípios → tabela acumulada: 851
   2023: 767 municípios → tabela acumulada: 851
   2024: 758 municípios → tabela acumulada: 851
   2025: 768 municípios → tabela acumulada: 852

✅ Tabela base: (852, 106)


In [7]:
# ── Etapa 3: Totais por ano ───────────────────────────────────────────────────
print("⚙️  Calculando totais...")
totais = {}

for ano in anos:
    cols = [f"{ano}_{faixa}" for faixa in ordem_faixas if f"{ano}_{faixa}" in tabela.columns]
    if not cols:
        continue
    totais[f"total_{ano}"] = tabela[cols].sum(axis=1).astype(int)

tabela = pd.concat([tabela, pd.DataFrame(totais)], axis=1)

# Reordena colunas finais: municipios, codigo, YYYY_faixas (todos os anos),
# depois todas as total_YYYY ao final
colunas_ordenadas = ["municipios", "codigo"]
for ano in anos:
    colunas_ordenadas += [f"{ano}_{faixa}" for faixa in ordem_faixas if f"{ano}_{faixa}" in tabela.columns]
colunas_ordenadas += [f"total_{ano}" for ano in anos if f"total_{ano}" in tabela.columns]

tabela = tabela[colunas_ordenadas].sort_values("codigo").reset_index(drop=True)
print(f"✅ Totais calculados. Shape: {tabela.shape}")

⚙️  Calculando totais...
✅ Totais calculados. Shape: (852, 119)


In [8]:
# ── Etapa 4: Enriquecimento com código IBGE completo e Regional de Saúde ─────
print("\n⚙️  Enriquecendo com código IBGE e Regional de Saúde...")

try:
    df_mun = pd.read_csv(MUNICIPIOS, dtype=str)

    if "MUNICÍPIO NORMALIZADO" not in df_mun.columns:
        print("⚠️  Coluna 'MUNICÍPIO NORMALIZADO' não encontrada na tabela de municípios. Etapa ignorada.")
    else:
        mapa_ibge = df_mun.set_index("MUNICÍPIO NORMALIZADO")["CÓDIGO IBGE"].to_dict()

        # Preenche código via nome normalizado
        tabela["codigo_ibge_mapeado"] = tabela["municipios"].map(mapa_ibge)
        tabela["codigo"] = tabela["codigo_ibge_mapeado"].fillna(tabela["codigo"])

        # Expande códigos curtos (< 7 dígitos) para o código IBGE completo
        cods_ibge = df_mun["CÓDIGO IBGE"].astype(str)
        mask_short = tabela["codigo"].notna() & (tabela["codigo"].str.len() < 7)

        def expand_codigo(c: str) -> str:
            """Expande um código curto para 7 dígitos se houver match único na tabela IBGE."""
            if not isinstance(c, str) or len(c) >= 7:
                return c
            matches = cods_ibge[cods_ibge.str.endswith(c)]
            return matches.iloc[0] if len(matches) == 1 else c

        if mask_short.any():
            tabela.loc[mask_short, "codigo"] = tabela.loc[mask_short, "codigo"].map(expand_codigo)
            print(f"   Códigos curtos expandidos: {int(mask_short.sum())}")

        tabela = tabela.drop(columns=["codigo_ibge_mapeado"])

        sem_7dig = (tabela["codigo"].str.len() < 7).sum()
        print(f"   Municípios com código < 7 dígitos após expansão: {sem_7dig}")

        # Regional de Saúde, via join pelo código IBGE completo
        mapa_regional = df_mun.set_index("CÓDIGO IBGE")["REGIONAL NORMALIZADA"].to_dict()
        tabela["regional"] = tabela["codigo"].map(mapa_regional)
        sem_regional = int(tabela["regional"].isna().sum())
        if sem_regional > 0:
            print(f"   ⚠️ {sem_regional} município(s) sem Regional de Saúde correspondente.")

        print(f"✅ Enriquecimento concluído.")

except FileNotFoundError:
    print(f"⚠️  Arquivo de municípios não encontrado: {MUNICIPIOS}. Código IBGE não foi enriquecido.")

# Padroniza os nomes finais de colunas: MUNICÍPIO, CÓDIGO IBGE, REGIONAL DE SAÚDE, YYYY_TOTAL
rename_final = {"municipios": "MUNICÍPIO", "codigo": "CÓDIGO IBGE", "regional": "REGIONAL DE SAÚDE"}
rename_final.update({f"total_{ano}": f"{ano}_TOTAL" for ano in anos})
tabela = tabela.rename(columns=rename_final)

# Reordena: MUNICÍPIO, CÓDIGO IBGE, REGIONAL DE SAÚDE, restante
colunas = tabela.columns.tolist()
colunas.remove("REGIONAL DE SAÚDE")
posicao = colunas.index("CÓDIGO IBGE") + 1
colunas.insert(posicao, "REGIONAL DE SAÚDE")
tabela = tabela[colunas]

print(f"\n✅ Tabela final: {tabela.shape}")


⚙️  Enriquecendo com código IBGE e Regional de Saúde...
   Municípios com código < 7 dígitos após expansão: 0
   ⚠️ 6 município(s) sem Regional de Saúde correspondente.
✅ Enriquecimento concluído.

✅ Tabela final: (852, 120)


/tmp/ipykernel_53253/2449278509.py:13: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  tabela["codigo_ibge_mapeado"] = tabela["municipios"].map(mapa_ibge)
/tmp/ipykernel_53253/2449278509.py:38: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  tabela["regional"] = tabela["codigo"].map(mapa_regional)


---
## 6. Inspeção Pós-Processamento

Verificação da tabela final antes de salvar: shape, amostra de linhas, distribuição de totais e valores ausentes.

In [9]:
display(Markdown(f"**Shape final:** {tabela.shape[0]} linhas × {tabela.shape[1]} colunas"))

display(Markdown("**Primeiras linhas:**"))
colunas_total = [c for c in tabela.columns if c.endswith("_TOTAL")]
cols_preview = ["MUNICÍPIO", "CÓDIGO IBGE", "REGIONAL DE SAÚDE"] + colunas_total[:5]
display(tabela[cols_preview].head(5))

# Estatísticas do total geral (soma de todas as colunas YYYY_TOTAL por município)
total_por_municipio = tabela[colunas_total].sum(axis=1)
total_geral = int(total_por_municipio.sum())
media       = total_por_municipio.mean()
mediana     = total_por_municipio.median()
desvio      = total_por_municipio.std()

print(f"\nTotal de exames no período : {total_geral:,}".replace(",", "."))
print(f"Média por município        : {media:,.0f}".replace(",", "."))
print(f"Mediana por município      : {mediana:,.0f}".replace(",", "."))
print(f"Desvio padrão              : {desvio:,.0f}".replace(",", "."))

# Valores ausentes
missing = tabela.isnull().sum()
missing = missing[missing > 0]
if not missing.empty:
    display(Markdown(f"**Colunas com valores ausentes:** {len(missing)}"))
    display(missing.head(10).to_frame("Nulos").style.background_gradient(cmap="Reds"))
else:
    display(Markdown("✅ **Sem valores ausentes.**"))

**Shape final:** 852 linhas × 120 colunas

**Primeiras linhas:**

,MUNICÍPIO,CÓDIGO IBGE,REGIONAL DE SAÚDE,2013_TOTAL,2014_TOTAL,2015_TOTAL,2016_TOTAL,2017_TOTAL
0,abadia dos dourados,3100104,uberlandia,0,155,84,84,445
1,abaete,3100203,sete lagoas,0,0,0,218,31
2,abre campo,3100302,manhumirim,0,2,231,301,382
3,acaiaca,3100401,ponte nova,0,0,0,0,0
4,acucena,3100500,coronel fabriciano,0,283,0,375,85



Total de exames no período : 4.300.516
Média por município        : 5.048
Mediana por município      : 1.426
Desvio padrão              : 22.621


**Colunas com valores ausentes:** 1

,Nulos
REGIONAL DE SAÚDE,6


---
## 7. Salvando o Resultado

In [10]:
tabela.to_csv(SAIDA, index=False, encoding="utf-8-sig")

print("╔══════════════════════════════════════════════════════╗")
print("║              Arquivo gerado com sucesso              ║")
print("╠══════════════════════════════════════════════════════╣")
print(f"║  Arquivo : {str(SAIDA.name):<42}║")
print(f"║  Linhas  : {len(tabela):<42}║")
print(f"║  Colunas : {len(tabela.columns):<42}║")
print(f"║  Anos    : {str(anos):<42}║")
print("╚══════════════════════════════════════════════════════╝")

╔══════════════════════════════════════════════════════╗
║              Arquivo gerado com sucesso              ║
╠══════════════════════════════════════════════════════╣
║  Arquivo : tabela_mamografia.csv                     ║
║  Linhas  : 852                                       ║
║  Colunas : 120                                       ║
║  Anos    : [2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]║
╚══════════════════════════════════════════════════════╝
